In [1]:
!pip -q install -U pip setuptools wheel
!pip -q install "transformers==4.56.2" "sentence-transformers==3.0.1" "huggingface-hub<1.0" "tokenizers==0.22.2"
!pip -q install langchain langchain-groq pypdf
!pip -q install llama-index llama-index-llms-groq llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [4]:
!pip -q install chromadb

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.26.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have

In [9]:
import os
import re
import json
import glob
import sqlite3
import getpass
from datetime import datetime
from typing import List, Dict, Any
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq


In [2]:
import getpass
import os

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

Enter API key for Groq: ··········


In [6]:
# ---------- PATHS ----------
DATA_DIR = "/content/drive/MyDrive/data"     # our RAG files folder
DB_PATH = "/content/agent_memory.sqlite"     # chat + episodic memory
CHROMA_DIR = "/content/chroma_agent_memory"  # semantic memory (vector DB)

# ---------- MODELS ----------
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("DATA_DIR:", DATA_DIR)
print("DB_PATH:", DB_PATH)
print("CHROMA_DIR:", CHROMA_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

DATA_DIR: /content/drive/MyDrive/data
DB_PATH: /content/agent_memory.sqlite
CHROMA_DIR: /content/chroma_agent_memory


###Create SQLite tables for checkpointed chat history and episodic memory

In [7]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS chat_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    role TEXT,
    content TEXT,
    created_at TEXT
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS episodes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    event_type TEXT,
    detail TEXT,
    created_at TEXT
)
""")

conn.commit()
conn.close()

print("SQLite tables created successfully.")

SQLite tables created successfully.


### Helper functions for session memory and episodic memory

In [8]:
def save_chat_message(session_id: str, role: str, content: str):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO chat_history (session_id, role, content, created_at) VALUES (?, ?, ?, ?)",
        (session_id, role, content, datetime.utcnow().isoformat())
    )
    conn.commit()
    conn.close()

def load_full_chat_history(session_id: str) -> List[Dict[str, str]]:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        "SELECT role, content FROM chat_history WHERE session_id = ? ORDER BY id ASC",
        (session_id,)
    )
    rows = cur.fetchall()
    conn.close()
    return [{"role": r[0], "content": r[1]} for r in rows]

def load_working_memory(session_id: str, last_k: int = 6) -> List[Dict[str, str]]:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        "SELECT role, content FROM chat_history WHERE session_id = ? ORDER BY id DESC LIMIT ?",
        (session_id, last_k)
    )
    rows = cur.fetchall()
    conn.close()
    rows = rows[::-1]
    return [{"role": r[0], "content": r[1]} for r in rows]

def save_episode(session_id: str, event_type: str, detail: str):
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO episodes (session_id, event_type, detail, created_at) VALUES (?, ?, ?, ?)",
        (session_id, event_type, detail, datetime.utcnow().isoformat())
    )
    conn.commit()
    conn.close()

def load_recent_episodes(session_id: str, limit: int = 5) -> List[Dict[str, str]]:
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        "SELECT event_type, detail FROM episodes WHERE session_id = ? ORDER BY id DESC LIMIT ?",
        (session_id, limit)
    )
    rows = cur.fetchall()
    conn.close()
    rows = rows[::-1]
    return [{"event_type": r[0], "detail": r[1]} for r in rows]

print("Memory helper functions ready.")

Memory helper functions ready.


### Load documents with LlamaIndex and split into nodes

In [10]:
# Load files from DATA_DIR using LlamaIndex
documents = SimpleDirectoryReader(DATA_DIR).load_data()
print(f"Loaded {len(documents)} documents")

# Split documents into semantically cleaner chunks
splitter = SentenceSplitter(chunk_size=800, chunk_overlap=100)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} nodes/chunks")

Loaded 156 documents
Created 334 nodes/chunks


### Build semantic memory in ChromaDB from LlamaIndex nodes

In [11]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

collection_name = "semantic_memory_demo"

try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(name=collection_name)

ids = []
documents_text = []
metadatas = []
embeddings = []

for i, node in enumerate(nodes):
    text = node.get_content().strip()
    meta = node.metadata or {}

    source = meta.get("file_name", "unknown")
    page = meta.get("page_label", meta.get("page_number", meta.get("page", "?")))

    emb = embed_model.encode(text).tolist()

    ids.append(f"node_{i}")
    documents_text.append(text)
    metadatas.append({
        "source": str(source),
        "page": str(page)
    })
    embeddings.append(emb)

if ids:
    collection.add(
        ids=ids,
        documents=documents_text,
        metadatas=metadatas,
        embeddings=embeddings
    )
    print(f"Indexed {len(ids)} nodes into ChromaDB.")
else:
    print("No nodes found to index.")

Indexed 334 nodes into ChromaDB.


### Semantic search function using ChromaDB

In [12]:
def semantic_search(query: str, top_k: int = 3) -> str:
    q_emb = embed_model.encode(query).tolist()

    result = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k
    )

    docs = result.get("documents", [[]])[0]
    metas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]

    blocks = []
    for doc, meta, dist in zip(docs, metas, distances):
        src = meta.get("source", "unknown")
        page = meta.get("page", "?")
        score = round(float(dist), 4)
        blocks.append(
            f"[SOURCE: {src} p.{page} | distance={score}]\n{doc[:500]}"
        )

    return "\n\n".join(blocks) if blocks else "No semantic memory found."

# Quick test
print(semantic_search("agentic RAG and memory", top_k=2))

[SOURCE: 2505.10468v4.pdf p.35 | distance=0.8816]
36, pp. 11809–11822, 2023.
[167] J. Guo, N. Li, J. Qi, H. Yang, R. Li, Y . Feng, S. Zhang, and M. Xu,
“Empowering working memory for large language model agents,”
arXiv preprint arXiv:2312.17259 , 2023.
[168] S. Agashe, J. Han, S. Gan, J. Yang, A. Li, and X. E. Wang, “Agent s:
An open agentic framework that uses computers like a human,” arXiv
preprint arXiv:2410.08164, 2024.
[169] C. DeChant, “Episodic memory in ai agents poses risks that should be
studied and mitigated,” arXiv preprint arXiv:25

[SOURCE: 2508.17692v1.pdf p.32 | distance=0.8957]
Although agent-based approaches have not
yet become mainstream in these fields, a growing body of pioneering work has begun to explore
the frontiers of their capabilities in complex tasks like Embodied Interaction, GUI Operation, and
Strategic Reasoning.
In Embodied Interaction, an agent’s reasoning process typically follows a perceive-plan-execute-
memory cycle. The primary goal in this field i

### Minimal demonstrations of all 4 memory types

In [13]:
session_id = "class_demo_session"

# Save some chat messages
save_chat_message(session_id, "user", "My topic is Agentic AI.")
save_chat_message(session_id, "assistant", "Okay, your topic is Agentic AI.")
save_chat_message(session_id, "user", "I taught vector databases today.")
save_chat_message(session_id, "assistant", "Noted. You taught vector databases.")
save_chat_message(session_id, "user", "Next lecture I will teach LangGraph.")
save_chat_message(session_id, "assistant", "Great. LangGraph is a natural next topic.")

print("=== 1) WORKING MEMORY ===")
print(load_working_memory(session_id, last_k=4))

print("\n=== 2) SESSION MEMORY / CHECKPOINTING IN SQLITE ===")
print(load_full_chat_history(session_id))

print("\n=== 3) EPISODIC MEMORY ===")
save_episode(session_id, "reflection", "Students understood the difference between working memory and vector DB.")
save_episode(session_id, "task_result", "Need a better example for episodic memory next class.")
print(load_recent_episodes(session_id, limit=5))

print("\n=== 4) SEMANTIC MEMORY IN CHROMADB ===")
print(semantic_search("What is agentic RAG?", top_k=2))

=== 1) WORKING MEMORY ===
[{'role': 'user', 'content': 'I taught vector databases today.'}, {'role': 'assistant', 'content': 'Noted. You taught vector databases.'}, {'role': 'user', 'content': 'Next lecture I will teach LangGraph.'}, {'role': 'assistant', 'content': 'Great. LangGraph is a natural next topic.'}]

=== 2) SESSION MEMORY / CHECKPOINTING IN SQLITE ===
[{'role': 'user', 'content': 'My topic is Agentic AI.'}, {'role': 'assistant', 'content': 'Okay, your topic is Agentic AI.'}, {'role': 'user', 'content': 'I taught vector databases today.'}, {'role': 'assistant', 'content': 'Noted. You taught vector databases.'}, {'role': 'user', 'content': 'Next lecture I will teach LangGraph.'}, {'role': 'assistant', 'content': 'Great. LangGraph is a natural next topic.'}]

=== 3) EPISODIC MEMORY ===
[{'event_type': 'reflection', 'detail': 'Students understood the difference between working memory and vector DB.'}, {'event_type': 'task_result', 'detail': 'Need a better example for episodic m

/tmp/ipykernel_194/3990493983.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (session_id, role, content, datetime.utcnow().isoformat())
/tmp/ipykernel_194/3990493983.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (session_id, event_type, detail, datetime.utcnow().isoformat())


### One combined mini-agent using all 4 memory types

In [14]:
def format_chat(messages: List[Dict[str, str]]) -> str:
    if not messages:
        return "None"
    return "\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages])

def format_episodes(episodes: List[Dict[str, str]]) -> str:
    if not episodes:
        return "None"
    return "\n".join([f"{e['event_type'].upper()}: {e['detail']}" for e in episodes])

def ask_agent(session_id: str, user_query: str, top_k: int = 3) -> str:
    # Save user query to session memory
    save_chat_message(session_id, "user", user_query)

    # 1. Working memory
    working_memory = load_working_memory(session_id, last_k=6)

    # 2. Semantic memory
    semantic_memory = semantic_search(user_query, top_k=top_k)

    # 3. Episodic memory
    episodic_memory = load_recent_episodes(session_id, limit=3)

    prompt = f"""
You are a teaching assistant for Agentic AI.

Use the following memory sources:

1. WORKING MEMORY (recent messages)
{format_chat(working_memory)}

2. SESSION MEMORY
The full conversation is checkpointed in SQLite.

3. SEMANTIC MEMORY (retrieved from ChromaDB)
{semantic_memory}

4. EPISODIC MEMORY (recent reflections and outcomes)
{format_episodes(episodic_memory)}

Answer the user's question clearly and simply.
Use semantic memory if relevant.
If not enough evidence is found, say that honestly.

USER QUESTION:
{user_query}
"""

    answer = llm.invoke(prompt).content.strip()

    save_chat_message(session_id, "assistant", answer)
    save_episode(session_id, "qa_turn", f"Q: {user_query} | A: {answer[:200]}")

    return answer

# Run final combined demo
response = ask_agent(
    "class_demo_session_2",
    "Explain the difference between working memory and semantic memory in agentic AI."
)
print(response)

/tmp/ipykernel_194/3990493983.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (session_id, role, content, datetime.utcnow().isoformat())


Based on our previous conversation and the information stored in our memory sources, I can explain the difference between working memory and semantic memory in agentic AI.

Working memory in agentic AI refers to the temporary storage and manipulation of information necessary for completing a task or solving a problem. It's like a mental workspace where the agent can hold and process information for a short period. This type of memory is essential for tasks that require reasoning, decision-making, and problem-solving.

Semantic memory, on the other hand, is a type of long-term memory that stores structured domain knowledge and facts. It's like a vast library of information that the agent can draw upon to inform its decisions and actions. Semantic memory is crucial for tasks that require understanding and applying domain-specific knowledge.

To illustrate the difference, consider a scenario where an agentic AI is tasked with completing a complex task, such as planning a route for a deliv

/tmp/ipykernel_194/3990493983.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (session_id, event_type, detail, datetime.utcnow().isoformat())
